# Bias-correct climate model data using the QDM method

The Quantile Delta Method, as summarized by [Gergel et al.](https://gmd.copernicus.org/articles/17/191/2024/): 
$$
\begin{aligned}
x_{m,p}^*(t) &= F_{o,h}^{-1}\left[\tau_{m,p}(t)\right] \\ 
&+ \left(x_{m,p}(t) - F_{m,h}^{-1}\left[\tau_{m,p}(t)\right]\right)
\end{aligned}
$$
which is reanalysis distribution at model quantiles + model change in those quantiles, since:

$$\begin{aligned}
x_{m,p}(t) &= \text{a future value} \\
x_{m,p}(t)^* &= \text{the future value BC'ed} \\
F_{o,h}^{-1} &= \text{reanalysis inverse cdf} \\
F_{m,h}^{-1} &= \text{future model inverse cdf} \\
\tau_{m,p}(t) &= \text{non-exceedence probability associated with }x_{m,p}(t)
\end{aligned}$$

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import fsspec
from tqdm.notebook import tqdm
import zarr
import dask
from functools import reduce
import os
from distributed import Client
import warnings

from funcs_support import get_filepaths, get_params, utility_save
from funcs_preprocessing import preprocess_models
from funcs_processing import calc_pctof_models, calc_rolling_quantiles, bias_correct_qdm, get_quantile_diffs
from funcs_aux import _create_filenames, _verify_gwl_range, _load_gwls, extract_gwl, _restore_doys

from numba import jit as njit

dir_list = get_params()

In [2]:
# Start dask client
client = Client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34177,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39323,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/41431/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:43293,


2026-02-15 08:46:02,938 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 3087704ddc0c367ecea78452ff176c3b initialized by task ('open_dataset-tas-rechunk-transfer-85b349260eb00bf97e67a91734958d31', 0, 0, 0, 15, 0, 0) executed on worker tcp://127.0.0.1:32963
2026-02-15 08:46:08,738 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 3087704ddc0c367ecea78452ff176c3b deactivated due to stimulus 'task-finished-1771170368.7376964'
2026-02-15 09:04:09,722 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 96650fd954afaa735a166bb101f02c5d initialized by task ('open_dataset-tas-rechunk-transfer-d22cdcd38399a9e75baa3ea50e853627', 0, 0, 0, 49, 0, 0) executed on worker tcp://127.0.0.1:39323
2026-02-15 09:04:16,541 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 96650fd954afaa735a166bb101f02c5d deactivated due to stimulus 'task-finished-1771171456.5405571'
2026-02-15 09:29:06,686 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 4a3eeffc19187700c4e7

In [3]:
#----------- Setup -----------
max_nruns = 100 #50 # Max # of runs to process per model... to allow a minimum processing first

params_var = {'var':'tas','freq':'day'}
#params_var = {'var':'tasmax','freq':'day'}

params_subset = {'lat':slice(-56,86),'lon':slice(-180,180)}
params_proc = {'wwidth':31,'nqs':100,'mod_rea':'MERRA2','bc_method':'qdm'}
gwl_info_rea = pd.Series({'warming_level':0.61,'start_year':1982,'end_year':2001})
# Get reference 1x1 grid
ref_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.5,
														 params_subset['lat'].stop-0.4)),
								'lon':(['lon'],np.arange(params_subset['lon'].start+0.5,
														 params_subset['lon'].stop-0.4))})

# Get filesystem
fs = fsspec.implementations.local.LocalFileSystem()

#----------- Processing Models -----------
# Get GWL info
gwl_info = _load_gwls()

# Get model filepath info
df = get_filepaths()
df = df.query('varname == "'+params_var['var']+'" and '+
              'freq == "'+params_var['freq']+'"')
# Get global files (no suffix)
if ('suffix' not in params_var) and 'suffix' in df.columns:
    df = df.where(df.suffix != df.suffix).dropna(how='all')

keys_all = [k for k in gwl_info.groupby(['model','ensemble','exp']).groups]
keys_all = [k for k in keys_all if k in df.groupby(['model','run','exp']).groups]
# Subset to just files that have both historical and future data 
# (saved under the future experiment name, not 'historical') 
keys_all = [k for k in keys_all if k[2] != 'historical']

if max_nruns is not None:
    # Process just a subset of keys if desired
    #keys_all = [tuple(row[1].values) for row in pd.DataFrame(keys_all).groupby(0).head(max_nruns).iterrows()]
    # And preferentially process higher SSP'ed keys, to increase
    # the number of runs that pass a given GWL
    keys_all = pd.MultiIndex.from_frame(pd.DataFrame(keys_all).sort_values([0,2],ascending=[True,False]).groupby([0]).head(max_nruns),
                         names = ['mod','run','exp'])

## Get CDFs of reference datasets

In [4]:
#----------- Preprocessing Reanalysis -----------
## Preprocess reanalysis
# (Regrid, rechunk, reshape into day-of-year x year)
for mod_rea in ['ERA5-025','MERRA2','JRA-3Q','GMFD']:
    preprocess_models(file_rows=df.query('model == "'+mod_rea+'"'),
                      gwl_info=gwl_info_rea,
                      params_var=params_var,
                      params_proc=params_proc,
                      params_subset=params_subset,
                      ref_grid=ref_grid,
                      fs=fs)


/glade/derecho/scratch/schwarzwald/bcd_me/ERA5-025/tas_day_ERA5-025_historical_reanalysis_CACHE-RESHAPE.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MERRA2/tas_day_MERRA2_historical_reanalysis_CACHE-RESHAPE.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/JRA-3Q/tas_day_JRA-3Q_historical_reanalysis_CACHE-RESHAPE.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/GMFD/tas_day_GMFD_historical_reanalysis_CACHE-RESHAPE.zarr already exists, skipped!


In [5]:
## Preprocess reanalysis
# (Regrid, rechunk, reshape)
# PRISM separately with a smaller time chunk size, because it's so darn huge
import os
if not os.path.exists(dir_list['proc']+'PRISM/tasmax_day_PRISM_historical_reanalysis_CACHE-RESHAPE.zarr'):
    for mod_rea in ['PRISM']:
        preprocess_models(file_rows=df.query('model == "'+mod_rea+'"'),
                          gwl_info=gwl_info_rea,
                          params_var=params_var,
                          params_proc=params_proc,
                          params_subset=params_subset,
                          ref_grid=ref_grid.sel(lat=slice(23,51),lon=slice(-126,-65.5)),
                          chunk_sizes = {'geo':20,'time':1},
                          fs=fs)
    # Silly, but _create_filenames assumes this form 
    os.system('mv '+dir_list['proc']+'PRISM/tasmax_day_PRISM_historical_obs_CACHE-RESHAPE.zarr '+
             dir_list['proc']+'PRISM/tasmax_day_PRISM_historical_reanalysis_CACHE-RESHAPE.zarr')
else:
    print(dir_list['proc']+'PRISM/tasmax_day_PRISM_historical_reanalysis_CACHE-RESHAPE.zarr exists, skipped!')

/glade/derecho/scratch/schwarzwald/bcd_me/PRISM/tasmax_day_PRISM_historical_reanalysis_CACHE-RESHAPE.zarr exists, skipped!


In [6]:
#----------- Get quantiles of reanalysis -----------
from funcs_processing import calc_quantiles
proc_mods = ['ERA5-025','MERRA2','JRA-3Q','GMFD']
if params_var['var'] == 'tasmax':
    proc_mods.append('PRISM')

for mod_rea in proc_mods:

    params_proc['mod_rea'] = mod_rea
    fns_out = _create_filenames(gwl_info_rea,params_var,params_proc)

    if not os.path.exists(fns_out['qrea']):
        # Load reanalysis
        ds_rea = xr.open_zarr(fns_out['reshp_rea'])
        
        # Change year to generic counter (e.g. 1:20)
        ds_rea['year'] = np.arange(1,ds_rea.sizes['year']+1)
        
        # Homogenize grids (if the original input `ds` doesn't fully cover
        # the `ref_grid`, it gets trimmed. Make sure the `ds_rea` also
        # gets trimmed to the same in that case
        #ds_rea = ds_rea.sel(lat=ds.lat,lon=ds.lon)
        
        # Get quantiles to fit
        qs = np.arange(1/params_proc['nqs']/2,
                       (1-1/params_proc['nqs']/2+(1/(params_proc['nqs']*10))),
                       1/params_proc['nqs'])
        
        
        # THIS SHOULD ACTUALLY BE DONE AT THE TOP, ONCE FOR EACH REANALYSIS
        dqs_rea = xr.apply_ufunc(calc_rolling_quantiles,
                                 ds_rea[params_var['var']],
                                 input_core_dims = [['dayofyear','year']],
                                 output_core_dims = [['dayofyear','quantile']],
                                 kwargs = {'wwidth':params_proc['wwidth'],'quantiles':qs},
                                 dask='parallelized',
                                     vectorize=True,
                                     dask_gufunc_kwargs={'output_sizes':{'quantile':params_proc['nqs']}},
                                     output_dtypes = [float]
                                     )
        
        utility_save(dqs_rea.drop_encoding(),fns_out['qrea'],save_kwargs={'zarr_format':3,'consolidated':False})
    else:
        print(fns_out['qrea']+' exists, skipped!')

/glade/derecho/scratch/schwarzwald/bcd_me/ERA5-025/tasq_day_ERA5-025_historical_reanalysis_19820101-20011231_GWL0-61.zarr exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MERRA2/tasq_day_MERRA2_historical_reanalysis_19820101-20011231_GWL0-61.zarr exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/JRA-3Q/tasq_day_JRA-3Q_historical_reanalysis_19820101-20011231_GWL0-61.zarr exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/GMFD/tasq_day_GMFD_historical_reanalysis_19820101-20011231_GWL0-61.zarr exists, skipped!


## QDM bias-correction

In [10]:
allow_grid_adjustment = True
grid_adj_reas = ['PRISM'] # Since only CONUS
chunk_sizes = {'geo':20}

overwrite = False
#----------- Processing by model/exp/run -----------
for keys in tqdm(keys_all):
    print('\n***********************\nProcessing '+', '.join(keys)+'!\n***********************')
    
    # GWL parameters for this model / exp / run
    gwl_info_tmp = gwl_info.reset_index().set_index(['model','ensemble','exp']).sort_index().loc[keys,:]
    # Local filepath parameters for this model / exp / run
    df_tmp = df.query('model == "'+keys[0]+'" and run == "'+keys[1]+'" and ('+
                      'exp == "historical" or exp == "'+keys[2]+'")')

    if len(df_tmp)==1:
        warnings.warn('Only one file found for '+', '.join(keys)+', skipping.')
        continue

    # Double-check that local files are present for all needed
    try:
        gwl_info_tmp = _verify_gwl_range(df_tmp,gwl_info_tmp,gwl_info_rea)
    except Exception as e:
        warnings.warn(str(e)+' Skipping.')
        continue

    for mod_rea in ['GMFD','ERA5-025','MERRA2','JRA-3Q']:
        params_proc['mod_rea'] = mod_rea
        
        fns_out = _create_filenames(gwl_info_rea,params_var,params_proc,file_rows=df_tmp)
    
        if overwrite or (not fs.exists(fns_out['biascrct_qdm']+'/.done')):
            # Remove intermediate files if existing
            if overwrite: 
                for file in ['reshp_mod','pctof','qdiff_mod']:
                    if fs.exists(fns_out[file]):
                        fs.rm(fns_out[file],recursive=True)
                        print(fns_out[file]+' removed to allow overwrite!')

            #----------- Preprocessing of model files -----------
            # Regrid, rechunk, reshape model 
            if not fs.exists(fns_out['reshp_mod']):
                # File is not deleted between reanalyses
                preprocess_models(file_rows=df_tmp,
                                  gwl_info=gwl_info_tmp,
                                  params_var=params_var,
                                  params_proc=params_proc,
                                  params_subset=params_subset,
                                  ref_grid=ref_grid,
                                  fs=fs)

            #----------- Quantile calculations -----------
            # Calculate rolling percentile of each value in models
            if not fs.exists(fns_out['pctof']):
                # File is not deleted between reanalyses
                calc_pctof_models(file_rows=df_tmp,
                                  gwl_info=gwl_info_tmp,
                                  gwl_info_rea=gwl_info_rea,
                                  params_var=params_var,
                                  params_proc=params_proc,
                                  fs=fs)

            #----------- Quantile change calculations -----------
            # This all should be something like calc_qdiffs_model
            # Calculate changes in quantiles in models between reference 
            if not fs.exists(fns_out['qdiff_mod']):
                # GWL and other GWLs
                # Load raw GCM output
                ds = xr.open_zarr(fns_out['reshp_mod'])

                # Force load coordinates (otherwise occasionally result in chunking
                # errors)
                for v in ds.coords:
                    ds[v] = ds[v].load()
                # Still can cause an error (e.g., for "time_bnds" coordinate in CESM2-WACCM
                # that isn't used in this processing); extract_gwl below can "unload" coordinates
                # So drop offending variables 
                ds = ds.drop_vars([v for v in ds.coords if v not in ['lat','lon','year','dayofyear']],
                                  errors='ignore')
                
                # Reshape to be by 20-year GWL chunks instead
                ds = xr.concat([extract_gwl(ds,row) for row in gwl_info_tmp.iterrows()],dim='gwl')
                
                # Get quantiles to fit
                qs = np.arange(1/params_proc['nqs']/2,
                               (1-1/params_proc['nqs']/2+(1/(params_proc['nqs']*10))),
                               1/params_proc['nqs'])
    
                from funcs_processing import calc_quantile_diffs_array
    
                # Apply the Numba function for sliding quantile differences
                dqs = xr.apply_ufunc(calc_quantile_diffs_array,
                                     ds[params_var['var']].sel(gwl=gwl_info_rea.warming_level),
                                     ds[params_var['var']],
                                     input_core_dims = [['dayofyear','year'],['dayofyear','year']],
                                     output_core_dims = [['dayofyear','quantile']],
                                     kwargs = {'wwidth':params_proc['wwidth'],'quantiles':qs},
                                     dask='parallelized',
                                     vectorize=True,
                                     dask_gufunc_kwargs={'output_sizes':{'quantile':params_proc['nqs']}},
                                     output_dtypes = [float]
                                     )
    
                chunk_sizes = {'geo':20}
    
                # Subset to the 365 days actually processed 
                dqs = _restore_doys(dqs,params_proc)
                
                # Turn back to dataset
                dqs = dqs.to_dataset(name='q'+params_var['var']+'diff')
                
                # Sometimes the chunks get messed up 
                dqs = dqs.chunk({'lat':chunk_sizes['geo'],'lon':chunk_sizes['geo']})
    
                # Save
                utility_save(dqs.drop_encoding(),fns_out['qdiff_mod'],save_kwargs = {'zarr_format':3},
                             zarr_mode = 'w')


            #----------- Bias correcting and output -----------
            # This all (except the top here) should be wrapper_biascorrect_qdm()
            if fs.exists(fns_out['biascrct_qdm']):
                fs.rm(fns_out['biascrct_qdm'],recursive=True)
                print(fns_out['biascrct_qdm']+' removed to allow overwrite!')
                
            # Load percentiles of model data
            ds_pctof = xr.open_zarr(fns_out['pctof'])
            
            # Load changes in quantiles
            dqs = xr.open_zarr(fns_out['qdiff_mod'])
            #dqs = -dqs # the test file is accidentally flipped. it has since been fixed.
            
            # Load raw GCM output
            dsf = xr.open_zarr(fns_out['reshp_mod'])

            # Force load coordinates (otherwise occasionally result in chunking
            # errors)
            for v in dsf.coords:
                dsf[v] = dsf[v].load()
            # Still can cause an error (e.g., for "time_bnds" coordinate in CESM2-WACCM
            # that isn't used in this processing); extract_gwl below can "unload" coordinates
            # So drop offending variables 
            dsf = dsf.drop_vars([v for v in dsf.coords if v not in ['lat','lon','year','dayofyear']],
                              errors='ignore')
            
            # Reshape to be by 20-year GWL chunks instead
            dsf = xr.concat([extract_gwl(dsf,row) for row in gwl_info_tmp.iterrows()],dim='gwl')
            
            # Remove extra DOYs
            dsf = _restore_doys(dsf,params_proc)
            
            # Load reanalysis cdf
            qs_rea = xr.open_zarr(fns_out['qrea'],consolidated=False)
            qs_rea = _restore_doys(qs_rea,params_proc)

            if allow_grid_adjustment:
                if mod_rea in grid_adj_reas:
                    all_coords = {coord:[d[coord].values for d in [qs_rea[params_var['var']],
                                     dsf[params_var['var']],
                                      dqs[f'q{params_var['var']}diff'],
                                      ds_pctof[params_var['var']]]]
                                  for coord in ['lat','lon']}
                    common_coords = {coord:reduce(np.intersect1d,all_coords[coord])
                                             for coord in all_coords}
                    
                    n_diff_coords = {coord:np.atleast_1d(np.unique([len(k) for k in all_coords[coord]])) for coord in all_coords}
                    len_common_coords = {coord:len(coords) for coord,coords in common_coords.items()}
                    
                    if np.any(np.array([len(k) for c,k in n_diff_coords.items()]) > 1):
                        print(f'Reducing grids to {str(len_common_coords)}-length common grid')
                    
                    qs_rea = qs_rea.sel(**common_coords)
                    dsf = dsf.sel(**common_coords)
                    dqs = dqs.sel(**common_coords)
                    ds_pctof = ds_pctof.sel(**common_coords)
            
            
            # Get QDM bias-correction
            dsf_out = xr.apply_ufunc(bias_correct_qdm,
                                     qs_rea[params_var['var']],
                                     dsf[params_var['var']],
                                     dqs[f'q{params_var['var']}diff'],
                                     ds_pctof[params_var['var']],
                                     input_core_dims = [['dayofyear','quantile'],
                                                        ['dayofyear','year'],
                                                        ['dayofyear','quantile'],
                                                        ['dayofyear','year']],
                                             output_core_dims = [['dayofyear','year']],
                                             dask='parallelized',
                                            vectorize=True,
                                            output_dtypes = [float])

            #--------- Save ---------
            # Turn to dataset
            dsf_out = dsf_out.to_dataset(name = params_var['var'])

            # Sometimes the chunks get messed up 
            dsf_out = dsf_out.chunk({'lat':chunk_sizes['geo'],'lon':chunk_sizes['geo']})

            # Get save attributes
            attrs = {'SOURCE':'bias_correct_qdm_fix.ipynb',
                     'DESCRIPTION':'QDM bias correction by GWL, using '+params_proc['mod_rea']+' as a reference base.',
                     'REF_DATA':params_proc['mod_rea']}
            
            # Output and save
            utility_save(dsf_out.drop_encoding(),
                         fns_out['biascrct_qdm'],save_kwargs = {'zarr_format':3,'consolidated':False},
                         zarr_mode = 'w')

            # Create a "done" flag in the zarr store
            open(fns_out['biascrct_qdm']+'/.done', 'w').close()
            
        else:
            print(fns_out['biascrct_qdm']+' already exists, skipped!')
    
    # Remove intermediate files
    for file in ['reshp_mod','pctof','qdiff_mod']:
        if fs.exists(fns_out[file]):
            fs.rm(fns_out[file],recursive=True)
    


  0%|          | 0/371 [00:00<?, ?it/s]


***********************
Processing ACCESS-ESM1-5, r10i1p1f1, ssp585!
***********************
Using GWLs 0.61, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0
/glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tas_day_ACCESS-ESM1-5_ssp585_r10i1p1f1_ALLGWLS_projQDM-baseGMFD.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tas_day_ACCESS-ESM1-5_ssp585_r10i1p1f1_ALLGWLS_projQDM-baseERA5-025.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tas_day_ACCESS-ESM1-5_ssp585_r10i1p1f1_ALLGWLS_projQDM-baseMERRA2.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tas_day_ACCESS-ESM1-5_ssp585_r10i1p1f1_ALLGWLS_projQDM-baseJRA-3Q.zarr already exists, skipped!

***********************
Processing ACCESS-ESM1-5, r1i1p1f1, ssp585!
***********************
Using GWLs 0.61, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0
/glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tas_day_ACCESS-ESM1-5_ssp585_r1i1p1f1_ALLGWLS_p

/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 2.0, 2.5, 3.0, 3.5 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 2.0, 2.5, 3.0, 3.5 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in


Using GWLs 0.61, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5
/glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tas_day_CNRM-ESM2-1_ssp370_r4i1p1f2_ALLGWLS_projQDM-baseGMFD.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tas_day_CNRM-ESM2-1_ssp370_r4i1p1f2_ALLGWLS_projQDM-baseERA5-025.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tas_day_CNRM-ESM2-1_ssp370_r4i1p1f2_ALLGWLS_projQDM-baseMERRA2.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tas_day_CNRM-ESM2-1_ssp370_r4i1p1f2_ALLGWLS_projQDM-baseJRA-3Q.zarr already exists, skipped!

***********************
Processing CNRM-ESM2-1, r5i1p1f2, ssp370!
***********************
Using GWLs 0.61, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5
/glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tas_day_CNRM-ESM2-1_ssp370_r5i1p1f2_ALLGWLS_projQDM-baseGMFD.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tas_day_CNRM-ESM2-1_ssp370_r5i

/glade/derecho/scratch/schwarzwald/tmp/ipykernel_98511/509875905.py:19: UserWarning: Only one file found for CanESM5, r23i1p1f1, ssp245, skipping.
  warnings.warn('Only one file found for '+', '.join(keys)+', skipping.')
/glade/derecho/scratch/schwarzwald/tmp/ipykernel_98511/509875905.py:19: UserWarning: Only one file found for CanESM5, r25i1p2f1, ssp245, skipping.
  warnings.warn('Only one file found for '+', '.join(keys)+', skipping.')


...

/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning

/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r25i1p1f1_ALLGWLS_projQDM-baseJRA-3Q.zarr already exists, skipped!

***********************
Processing MIROC6, r26i1p1f1, ssp245!
***********************
Using GWLs 0.61, 1.0
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r26i1p1f1_ALLGWLS_projQDM-baseGMFD.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r26i1p1f1_ALLGWLS_projQDM-baseERA5-025.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r26i1p1f1_ALLGWLS_projQDM-baseMERRA2.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r26i1p1f1_ALLGWLS_projQDM-baseJRA-3Q.zarr already exists, skipped!

***********************
Processing MIROC6, r27i1p1f1, ssp245!
***********************
Using GWLs 0.61, 1.0
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r27i1p1f1_ALLGWLS_projQ

/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in


/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r5i1p1f1_ALLGWLS_projQDM-baseERA5-025.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r5i1p1f1_ALLGWLS_projQDM-baseMERRA2.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r5i1p1f1_ALLGWLS_projQDM-baseJRA-3Q.zarr already exists, skipped!

***********************
Processing MIROC6, r6i1p1f1, ssp245!
***********************
Using GWLs 0.61, 1.0
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r6i1p1f1_ALLGWLS_projQDM-baseGMFD.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r6i1p1f1_ALLGWLS_projQDM-baseERA5-025.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp245_r6i1p1f1_ALLGWLS_projQDM-baseMERRA2.zarr already exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MIROC6/tas_day_MIROC6_ssp24

/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0, 2.5 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0, 2.5 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
